# IrriGator — Full End-to-End Pipeline

Central notebook covering **every block** from raw data to irrigation recommendation.

| Block | What | Data source | Run frequency |
|-------|------|-------------|---------------|
| 0 | Data ingestion | ERA5-Land, COMEPHORE, Sentinel-2, soil, DEM | Once (historical archive) |
| 1 | Static layers | EU-SoilHydroGrids, IGN DEM | Once per parcel |
| 2 | Atmospheric forcing | ERA5-Land daily → parcel | Once (cache) |
| 3+4 | Simulation | Water balance + crop growth | Daily (up to today) |
| 5a | Short-term forecast | AROME (48h) + IFS ENS (15d, 51 members) | Daily |
| 5b | Seasonal outlook | SEAS5 (6 months, 51 members) | Monthly (optional) |
| 6 | Decision | Ensemble-aware irrigation recommendation | Daily |

---
## Configuration — set your "today" and paths here

In [ ]:
import logging
import os
from datetime import date
from pathlib import Path

# === EDIT THESE ===
REPO_ROOT = os.path.dirname(os.getcwd())  # adjust if needed
os.chdir(REPO_ROOT)

TODAY = date(2002, 8, 15)          # the date we treat as "today"
SIMULATION_START = date(2002, 1, 1) # start of the growing season simulation
LAST_IRRIGATION = None              # date of last irrigation, or None for rainfed
RUN_SEAS5 = False                   # set True to include seasonal outlook (slow)

REGION_CONFIG = "configs/dordogne.yaml"
PARCEL_CONFIG = "configs/parcels/example.yaml"
CROP_PARAMS_PATH = "configs/crop_parameters.yaml"
MODEL_PARAMS_PATH = "configs/model_parameters.yaml"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Repo root: {REPO_ROOT}")
print(f"Today:     {TODAY}")

---
## Block 0 — Data Ingestion

Run these cells **once** to download and cache the historical archive.
Skip if data is already on disk.

In [ ]:
from irrigator.config import load_region_config, load_parcel_config

cfg = load_region_config(REGION_CONFIG)
parcel = load_parcel_config(PARCEL_CONFIG)

print(f"Region: {cfg.name} (dept {cfg.code_departement})")
print(f"Parcel: {parcel.name} ({parcel.id})")
print(f"Crop:   {parcel.crop.get('type')} — planting {parcel.crop.get('planting_date')}")

In [ ]:
# --- ERA5-Land download (skip if already done) ---
# from irrigator.ingestion.cds_client import fetch_era5_land_range
# fetch_era5_land_range(cfg, date(2000, 1, 1), date(2003, 12, 31), include_validation=True)

In [ ]:
# --- Build static layers (skip if already done) ---
# from irrigator.ingestion.esdac_loader import load_soil_hydro, save_soil_hydro
# from irrigator.ingestion.ign_loader import load_terrain, save_terrain
# from irrigator.ingestion.grid import build_grid
#
# grid = build_grid(cfg)
# soil_grid = load_soil_hydro(cfg, grid)
# save_soil_hydro(soil_grid, cfg.processed_dir / "static")
# terrain_grid = load_terrain(cfg, grid)
# save_terrain(terrain_grid, cfg.processed_dir / "static")

---
## Block 1 — Static Layers (parcel-level)

Extract soil and terrain at the parcel location. Run once per parcel, cache.

In [ ]:
from irrigator.static_layers.soil import get_soil_profile
from irrigator.static_layers.terrain import get_terrain_params, get_era5_elevation

soil = get_soil_profile(cfg, parcel)
terrain = get_terrain_params(cfg, parcel)
terrain.era5_elevation_m = get_era5_elevation(cfg, parcel)

print(f"Soil AWC:  {soil.total_awc_mm:.0f} mm (source: {soil.source})")
print(f"Elevation: {terrain.elevation_m:.0f} m (ERA5 cell: {terrain.era5_elevation_m:.0f} m)")
print(f"Slope:     {terrain.slope_deg:.1f}°")

---
## Block 2 — Atmospheric Forcing

Process ERA5-Land hourly → daily (once), then extract at parcel with downscaling.

In [ ]:
from irrigator.atmospheric.era5_processor import load_daily, process_era5_to_daily, save_daily
from irrigator.atmospheric.forcing import extract_parcel_forcing

# First time: process hourly → daily (slow, do once)
# from irrigator.ingestion.cds_client import open_era5_land
# era5_hourly = open_era5_land(cfg)
# era5_daily = process_era5_to_daily(era5_hourly)
# save_daily(era5_daily, cfg)

# After that: just load cached
era5_daily = load_daily(cfg)
forcing = extract_parcel_forcing(era5_daily, parcel, terrain)

print(f"Forcing: {forcing.n_days} days")
print(f"T range: [{forcing.t_min.min():.1f}, {forcing.t_max.max():.1f}] °C")
print(f"Total precip: {forcing.precip_mm.sum():.0f} mm")

---
## Blocks 3+4 — Historical Simulation up to TODAY

Coupled crop growth + water balance from `SIMULATION_START` to `TODAY`.
The final state is the starting point for the forecast.

In [ ]:
import matplotlib.pyplot as plt
from irrigator.crop.phenology import CropParams
from irrigator.water_balance.bucket_model import run_simulation, states_to_dataframe

crop_params = CropParams.from_config(parcel.crop, crop_params_path=CROP_PARAMS_PATH)

historical_forcing = forcing.slice(SIMULATION_START, TODAY)

states = run_simulation(
    forcing=historical_forcing,
    parcel=parcel,
    terrain=terrain,
    soil=soil,
    crop_params=crop_params,
)

current_state = states[-1]

print(f"\n{'='*50}")
print(f"STATE ON {TODAY}")
print(f"{'='*50}")
print(f"Depletion:  {current_state.depletion:.0f} / {current_state.taw:.0f} mm")
print(f"Stress Ks:  {current_state.stress_coeff:.3f}")
print(f"Crop stage: {current_state.crop_stage}")
print(f"GDD:        {current_state.gdd:.0f}")
print(f"Root depth: {current_state.z_root:.2f} m")
print(f"Kc:         {current_state.kc:.2f}")

In [ ]:
# Plot historical simulation
df_hist = states_to_dataframe(states)

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

axes[0].fill_between(df_hist.index, 0, df_hist["taw"], alpha=0.15, color="blue", label="TAW")
axes[0].fill_between(df_hist.index, 0, df_hist["raw"], alpha=0.15, color="green", label="RAW")
axes[0].plot(df_hist.index, df_hist["depletion"], color="red", lw=1.5, label="Depletion")
axes[0].set_ylabel("mm")
axes[0].set_title("Soil water depletion vs available water")
axes[0].legend()

axes[1].bar(df_hist.index, df_hist["precip"], color="steelblue", alpha=0.7, label="Precip")
axes[1].plot(df_hist.index, df_hist["et0"], color="orange", lw=0.8, label="ET0")
axes[1].plot(df_hist.index, df_hist["etc_act"], color="red", lw=0.8, label="ETc actual")
axes[1].set_ylabel("mm/day")
axes[1].set_title("Water fluxes")
axes[1].legend()

axes[2].plot(df_hist.index, df_hist["stress_coeff"], color="darkred", lw=1.2)
axes[2].axhline(0.9, color="orange", ls="--", alpha=0.5, label="Stress threshold")
axes[2].set_ylabel("Ks")
axes[2].set_title("Water stress coefficient (1=none, 0=wilting)")
axes[2].set_ylim(0, 1.05)
axes[2].legend()

axes[3].plot(df_hist.index, df_hist["kc"], color="green", lw=1.2, label="Kc")
ax3b = axes[3].twinx()
ax3b.plot(df_hist.index, df_hist["gdd"], color="gray", ls="--", lw=0.8, label="GDD")
ax3b.set_ylabel("GDD (°C·days)", color="gray")
axes[3].set_ylabel("Kc")
axes[3].set_title("Crop coefficient and GDD accumulation")
axes[3].legend(loc="upper left")
ax3b.legend(loc="upper right")

plt.suptitle(f"Historical simulation: {SIMULATION_START} → {TODAY}", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Block 5a — Operational forecast pipeline

- **ERA5-Land** (Jan 1 -> 5 days ago): reanalysis, cached
- **AROME** rolling analysis (gap -> today): near real-time
- **AROME** (0-48h, deterministic, 1.3 km): deterministic, high-res
- **IFS ENS** (0-15d, 51 members, 0.25°): ensemble, probabilistic
- Blended: days 0-2 from AROME, days 3+ from IFS ENS

⚠️ For historical dates (TODAY in the past), we skip live forecast fetching
and use ERA5-Land as a stand-in to demonstrate the pipeline.

In [ ]:
# Three data sources stitched together:
#   1. ERA5-Land (Jan 1 → ~5 days ago)     — reanalysis, cached
#   2. AROME rolling analysis (gap → today) — near real-time
#   3. AROME 48h forecast (today → +2d)     — deterministic, high-res
#   4. IFS ENS forecast (today → +15d)      — ensemble, probabilistic
#      (AROME overrides IFS ENS for days 0-2)

from datetime import timedelta
import pandas as pd
from irrigator.atmospheric.forcing import extract_parcel_forcing

# --- Determine ERA5-Land last available day ---
era5_last = pd.Timestamp(era5_daily.valid_time.values[-1]).date()
print(f"ERA5-Land last day: {era5_last}")
print(f"Today:              {TODAY}")
print(f"Gap to fill:        {(TODAY - era5_last).days - 1} days")


In [ ]:
# --- Step 1: ERA5-Land historical (already done in Block 2) ---
era5_forcing = forcing.slice(SIMULATION_START, era5_last)
print(f"ERA5-Land forcing: {era5_forcing.n_days} days")


In [ ]:
# --- Step 2: AROME rolling analysis fills the gap ---
# Each day: fetch 00Z AROME run, take steps 0-23h, aggregate to daily
from irrigator.ingestion.meteofrance_client import fetch_forecast, open_forecast
from irrigator.forecasts.short_term import standardize_forecast_to_era5_format
from irrigator.atmospheric.era5_processor import process_era5_to_daily
import xarray as xr

gap_start = era5_last + timedelta(days=1)
gap_end = TODAY - timedelta(days=1)  # yesterday

# For HISTORICAL testing (TODAY in the past), use ERA5-Land as gap stand-in:
if gap_start <= gap_end:
    gap_forcing = forcing.slice(gap_start, gap_end)
    print(f"Gap filled: {gap_forcing.n_days} days ({gap_start} → {gap_end})")
else:
    gap_forcing = None
    print("No gap to fill")

# For LIVE operation, replace the above with:
# gap_daily_list = []
# for d_offset in range((gap_end - gap_start).days + 1):
#     gap_date = gap_start + timedelta(days=d_offset)
#     path = fetch_forecast(cfg, "arome", gap_date, run_hour=0)
#     raw = open_forecast(path)
#     daily = standardize_forecast_to_era5_format(raw, source="arome")
#     # Take only steps for this calendar day (0-23h from 00Z run)
#     daily = daily.sel(valid_time=daily.valid_time.dt.date == gap_date)
#     gap_daily_list.append(daily)
# gap_ds = xr.concat(gap_daily_list, dim="valid_time")
# gap_forcing = extract_parcel_forcing(gap_ds, parcel, terrain)


In [ ]:
# --- Step 3: AROME 48h deterministic forecast (today → +2d) ---
# For LIVE:
# arome_path = fetch_forecast(cfg, "arome", TODAY, run_hour=0)
# arome_raw = open_forecast(arome_path)
# arome_daily = standardize_forecast_to_era5_format(arome_raw, source="arome")
# arome_forcing_48h = extract_parcel_forcing(arome_daily, parcel, terrain)

# For HISTORICAL testing: use ERA5-Land as stand-in
arome_forcing_48h = forcing.slice(TODAY, TODAY + timedelta(days=1))
print(f"AROME forecast: {arome_forcing_48h.n_days} days")


In [ ]:
# --- Step 4: Stitch everything → run historical simulation ---
# ERA5-Land + gap + AROME today
full_forcing = era5_forcing
if gap_forcing:
    full_forcing = full_forcing.concat(gap_forcing)
full_forcing = full_forcing.concat(arome_forcing_48h)

print(f"Full forcing: {full_forcing.n_days} days ({SIMULATION_START} → ...)")

# Run simulation up to current state
states = run_simulation(
    forcing=full_forcing.slice(SIMULATION_START, TODAY),
    parcel=parcel,
    terrain=terrain,
    soil=soil,
    crop_params=crop_params,
)
current_state = states[-1]
print(
    f"\nState on {TODAY}: Dr={current_state.depletion:.0f} mm, Ks={current_state.stress_coeff:.3f}"
)


In [ ]:
from irrigator.forecasts.short_term import run_forward_balance
# --- Step 5: AROME deterministic forward balance (days 0-2) ---
arome_states = run_forward_balance(
    current_state,
    arome_forcing_48h,
    parcel,
    terrain,
    soil,
    crop_params,
)
print("AROME deterministic (days 0-2):")
for s in arome_states:
    print(f"  {s.date}: Ks={s.stress_coeff:.3f}, Dr={s.depletion:.0f} mm")


In [ ]:
# --- Step 6: IFS ENS ensemble forward balance (days 0-15) ---

from irrigator.forecasts.short_term import DailyForcing

# For LIVE:
# from irrigator.ingestion.ifs_ens_client import fetch_latest_ifs_ens, open_ifs_ens, process_ifs_ens_to_daily
# grib = fetch_latest_ifs_ens(cfg)
# ifs_daily = process_ifs_ens_to_daily(open_ifs_ens(grib, cfg))
# member_forcings = {}
# for m in ifs_daily.number.values:
#     member_forcings[int(m)] = extract_parcel_forcing(ifs_daily.sel(number=m), parcel, terrain)


# For HISTORICAL testing: synthetic ensemble (same as before)
import numpy as np

rng = np.random.default_rng(42)
n_members = 20
forecast_end = TODAY + timedelta(days=10)
pseudo_forcing = forcing.slice(TODAY + timedelta(days=1), forecast_end)

member_forcings = {}
for m in range(n_members):
    member_forcings[m] = DailyForcing(
        dates=pseudo_forcing.dates.copy(),
        t_min=pseudo_forcing.t_min + rng.normal(0, 1.0, pseudo_forcing.n_days),
        t_max=pseudo_forcing.t_max + rng.normal(0, 1.0, pseudo_forcing.n_days),
        t_mean=pseudo_forcing.t_mean + rng.normal(0, 0.8, pseudo_forcing.n_days),
        dewpoint=pseudo_forcing.dewpoint + rng.normal(0, 0.5, pseudo_forcing.n_days),
        wind_speed_2m=np.clip(
            pseudo_forcing.wind_speed_2m + rng.normal(0, 0.3, pseudo_forcing.n_days), 0, None
        ),
        pressure_kpa=pseudo_forcing.pressure_kpa.copy(),
        rs_mj=np.clip(pseudo_forcing.rs_mj + rng.normal(0, 2, pseudo_forcing.n_days), 0, None),
        precip_mm=np.clip(
            pseudo_forcing.precip_mm * rng.lognormal(0, 0.5, pseudo_forcing.n_days), 0, None
        ),
    )


In [ ]:
# --- Step 7: Blended ensemble stress report ---
# AROME overrides days 0-2, IFS ENS provides ensemble spread for days 3+
from irrigator.forecasts.short_term import run_ensemble_forward_balance

stress_report = run_ensemble_forward_balance(
    current_state=current_state,
    member_forcings=member_forcings,
    parcel=parcel,
    terrain=terrain,
    soil=soil,
    crop_params=crop_params,
    arome_states=arome_states,
    arome_days=2,  # AROME covers 48h = 2 days
)


In [ ]:
# --- Plot ensemble forecast ---
import pandas as pd

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

dates = [d.date for d in stress_report.daily_stats]
sources = [d.source for d in stress_report.daily_stats]

# Ks with ensemble spread
ax = axes[0]
ks_med = [d.ks_median for d in stress_report.daily_stats]
ks_p25 = [d.ks_p25 for d in stress_report.daily_stats]
ks_p75 = [d.ks_p75 for d in stress_report.daily_stats]
ks_min = [d.ks_min for d in stress_report.daily_stats]
ks_max = [d.ks_max for d in stress_report.daily_stats]

ax.fill_between(dates, ks_min, ks_max, alpha=0.1, color="red", label="min-max")
ax.fill_between(dates, ks_p25, ks_p75, alpha=0.3, color="red", label="p25-p75")
ax.plot(dates, ks_med, color="darkred", lw=2, marker="o", markersize=4, label="median Ks")
ax.axhline(0.9, color="orange", ls="--", alpha=0.5, label="stress threshold")
# Mark AROME vs IFS ENS
for i, src in enumerate(sources):
    if src == "arome":
        ax.axvspan(dates[max(0,i)], dates[min(i,len(dates)-1)], alpha=0.05, color="blue")
ax.set_ylabel("Ks")
ax.set_title("Water stress forecast — AROME (blue shade) + IFS ENS (ensemble spread)")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")

# Stress probability
ax = axes[1]
stress_prob = [d.stress_probability * 100 for d in stress_report.daily_stats]
colors = ["steelblue" if s == "arome" else "coral" for s in sources]
ax.bar(dates, stress_prob, color=colors, alpha=0.7)
ax.axhline(70, color="red", ls="--", alpha=0.5, label="Irrigate trigger (70%)")
ax.axhline(40, color="orange", ls="--", alpha=0.5, label="Watch trigger (40%)")
ax.set_ylabel("Stress probability (%)")
ax.set_title("Fraction of ensemble members showing stress")
ax.legend()

# Precipitation
ax = axes[2]
pr_mean = [d.precip_mean for d in stress_report.daily_stats]
pr_p25 = [d.precip_p25 for d in stress_report.daily_stats]
pr_p75 = [d.precip_p75 for d in stress_report.daily_stats]
ax.fill_between(dates, pr_p25, pr_p75, alpha=0.3, color="steelblue", label="p25-p75")
ax.plot(dates, pr_mean, color="darkblue", lw=1.5, marker="o", markersize=3, label="mean precip")
ax.set_ylabel("mm/day")
ax.set_title("Ensemble precipitation forecast")
ax.legend()

plt.suptitle(f"Forecast from {TODAY}: AROME (days 0-2) + IFS ENS (days 3+)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Block 5b — Seasonal Outlook (SEAS5, optional)

PCA analog matching → daily disaggregation → ensemble water balance.
Set `RUN_SEAS5 = True` at the top to enable.

In [ ]:
outlook = None

if RUN_SEAS5:
    from irrigator.ingestion.cds_client import fetch_seas5, open_seas5
    from irrigator.forecasts.seas5_processor import (
        build_era5_monthly_climatology,
        correct_seas5_monthly,
    )
    from irrigator.forecasts.disaggregation import generate_seasonal_scenarios
    from irrigator.decision.seasonal_outlook import compute_seasonal_outlook

    init_year = TODAY.year
    init_month = TODAY.month

    # Fetch + bias correct SEAS5
    fetch_seas5(cfg, year=init_year, month=init_month)
    seas5_anom = open_seas5(cfg, year=init_year, month=init_month)
    era5_clim = build_era5_monthly_climatology(cfg, start_year=1993, end_year=2016)
    seas5_corrected = correct_seas5_monthly(seas5_anom, era5_clim, init_month=init_month)

    # Analog disaggregation (PCA + Mahalanobis + spatial voting)
    scenarios = generate_seasonal_scenarios(
        cfg=cfg,
        era5_daily=era5_daily,
        seas5_corrected=seas5_corrected,
        init_month=init_month,
        n_leads=4,
    )

    # Run water balance under each scenario
    outlook = compute_seasonal_outlook(
        scenarios=scenarios,
        parcel=parcel,
        terrain=terrain,
        soil=soil,
        crop_params=crop_params,
    )

    print(f"\nSeasonal outlook ({outlook.n_members} members):")
    print(f"  Irrigation needed: {outlook.total_irrigation_p25:.0f}–{outlook.total_irrigation_p75:.0f} mm (p25–p75)")
    print(f"  Median: {outlook.total_irrigation_median:.0f} mm")
    print(f"  Stress days: median {outlook.stress_days_median:.0f}")
    print(f"  {outlook.season_description}")
else:
    print("SEAS5 seasonal outlook skipped (set RUN_SEAS5=True to enable)")

---
## Block 6 — Irrigation Decision

Combines current soil state + ensemble forecast → actionable recommendation.

In [ ]:
from irrigator.decision.rules import compute_recommendation, DecisionParams
from irrigator.decision.output import build_parcel_report, save_report, format_sms

params = DecisionParams.from_config(
    model_params_path=MODEL_PARAMS_PATH,
    parcel=parcel,
)

# Ensemble-aware recommendation (uses stress_report from Block 5a)
advice = compute_recommendation(
    current_state=current_state,
    stress_report=stress_report,
    crop_params=crop_params,
    params=params,
    last_irrigation_date=LAST_IRRIGATION,
)

print(f"\n{'='*60}")
print(f"IRRIGATION RECOMMENDATION — {TODAY}")
print(f"{'='*60}")
print(f"Irrigate:    {'YES' if advice.irrigate else 'NO'}")
if advice.irrigate:
    print(f"Dose:        {advice.dose_mm:.0f} mm")
    print(f"When:        {advice.recommended_date}")
print(f"Confidence:  {advice.confidence}")
if advice.stress_probability is not None:
    print(f"Stress prob: {advice.stress_probability:.0%}")
print(f"Reason:      {advice.reason}")

In [ ]:
# Build and save JSON report
report = build_parcel_report(
    parcel_id=parcel.id,
    today=TODAY,
    advice=advice,
    outlook=outlook,
)

import json
print(json.dumps(report, indent=2, default=str))

In [ ]:
# Save to disk
save_report(report, Path("output/"))

# SMS format
print(f"\nSMS: {format_sms(advice)}")

---
## Summary: data flow

```
ERA5-Land historical ──[Block 2: downscale]──→ DailyForcing
                                                     │
                                    [Blocks 3+4: simulation]──→ current_state
                                                                      │
AROME 0-48h ──[standardize]──[Block 2: downscale]──→ DailyForcing     │
                                                         │            │
IFS ENS 0-15d ──[process]──[Block 2: downscale]──→ 51× DailyForcing  │
    (51 members)                                         │            │
                                             [Block 5a: ensemble]─────┤
                                                    │                 │
                                          stress_report               │
                                                    │                 │
SEAS5 1-6mo ──[delta correct]──[PCA analog]──[extract 0.1°]          │
    (51 members)    ──[Block 2: downscale]──→ 51× seasonal forcing   │
                                             [Block 5b: seasonal]     │
                                                    │                 │
                                              outlook                 │
                                                    │                 │
                                        [Block 6: decision]───────────┘
                                                    │
                                              JSON report
                                              SMS message
```